# FastText + Word2Vec with SVM/Logistic Regression

### Import necessary libraries

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import re
import os
import sys
import codecs
import os.path
from sklearn.model_selection import train_test_split
from gensim.models import FastText, KeyedVectors
from nltk.tag.crf import CRFTagger
import pycrfsuite
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings("ignore")
from gensim.models import Word2Vec
from scipy.sparse import hstack, csr_matrix

sys.path.append(os.path.abspath(".."))

from preprocessing.utils import *

### Evaluation function

F1, AUC, AP

In [2]:
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report
)

def evaluate(y_true, y_probs, model_name="Model", threshold=0.5):
    """
    y_true  : ground truth labels (1=Mitterrand, 0=Chirac)
    y_probs : predicted P(Mitterrand)
    """
    y_preds = (y_probs >= threshold).astype(int)

    f1  = f1_score(y_true, y_preds, pos_label=1)
    auc = roc_auc_score(y_true, y_probs)
    ap  = average_precision_score(y_true, y_probs, pos_label=1)

    print(f"\n{'─'*40}")
    print(f"  {model_name}")
    print(f"{'─'*40}")
    print(f"  F1  (Mitterrand): {f1:.4f}")
    print(f"  AUC (ROC):        {auc:.4f}")
    print(f"  AP  (PR curve):   {ap:.4f}")
    print(f"{'─'*40}")
    print(classification_report(y_true, y_preds, target_names=["Chirac","Mitterrand"]))
    return {"f1": f1, "auc": auc, "ap": ap}

In [3]:
FILE_NAME = "../../data/corpus.tache1.learn.utf8"

alltxts, alllabs = load_pres(FILE_NAME)

print(len(alltxts))
print(alltxts[10])
print(alllabs[10])

57413
 A Brazzaville, que l'Afrique de demain se dessine.

1


### Load Text

p > 0.5 → Mitterrand

1 -> Mitterrand
0 -> Chirac

In [ ]:
# Chargement des données:
def load_pres(fname):
    alltxts = []
    alllabs = []
    s=codecs.open(fname, 'r','utf-8') # pour régler le codage
    while True:
        txt = s.readline()
        if(len(txt))<5:
            break
        #
        lab = re.sub(r"<[0-9]*:[0-9]*:(.)>.*","\\1",txt)
        txt = re.sub(r"<[0-9]*:[0-9]*:.>(.*)","\\1",txt)
        if lab.count('M') > 0:
            alllabs.append(1)   # Mitterrand = 1
        else:
            alllabs.append(0)   # Chirac = 0
        alltxts.append(txt)
    return alltxts,alllabs

In [5]:
fname = "../../data/corpus.tache1.learn.utf8"
alltxts, alllabs = load_pres(fname)

len(alltxts), len(alllabs)

(57413, 57413)

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    alltxts, alllabs,
    test_size=0.2,
    random_state=42,
    stratify=alllabs      # preserve 87/13 ratio in both splits
)

print(f"Train: {len(X_train)} sentences | Val: {len(X_val)} sentences")
print(f"Train Mitterrand: {sum(y_train)} ({100*sum(y_train)/len(y_train):.1f}%)")

Train: 45930 sentences | Val: 11483 sentences
Train Mitterrand: 6018 (13.1%)


### Train FastText on train corpus only

In [7]:
corpus = [text.lower().split() for text in X_train]

print("\nTraining FastText...")
ft_model = FastText(
    sentences=corpus,
    vector_size=200,
    window=7,
    min_count=2,
    workers=4,
    epochs=15,
    sg=1,           # Skip-gram: better for rare/specialized political vocabulary
)
print("FastText done.")


Training FastText...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


FastText done.


### Train Word2Vec (GloVe-style CBOW) on same corpus

In [8]:
print("Training Word2Vec...")
w2v_model = Word2Vec(
    sentences=corpus,       # same corpus as FastText (train only)
    vector_size=200,
    window=7,
    min_count=2,
    workers=4,
    epochs=15,
    sg=0,                   # CBOW, closer to GloVe's objective
)
print("Word2Vec done.")

Training Word2Vec...
Word2Vec done.


### Combined vectorizer

In [9]:
def vectorize_combined(text, ft_model, w2v_model):
    words = text.lower().split()
    if not words:
        return np.zeros(ft_model.vector_size + w2v_model.vector_size)
    
    ft_vecs  = np.array([ft_model.wv[w] for w in words])   # FastText handles OOV
    
    # Word2Vec has no subword → skip OOV words
    w2v_vecs = np.array([
        w2v_model.wv[w] for w in words if w in w2v_model.wv
    ])
    if len(w2v_vecs) == 0:
        w2v_vecs = np.zeros((1, w2v_model.vector_size))

    ft_mean  = ft_vecs.mean(axis=0)
    w2v_mean = w2v_vecs.mean(axis=0)

    return np.concatenate([ft_mean, w2v_mean])   # 200 + 200 = 400 dims


In [10]:
print("Vectorizing with FastText + Word2Vec...")
X_train_combined = np.array([vectorize_combined(t, ft_model, w2v_model) for t in X_train])
X_val_combined   = np.array([vectorize_combined(t, ft_model, w2v_model) for t in X_val])
print(f"Combined feature shape: {X_train_combined.shape}")  # (n, 400)


Vectorizing with FastText + Word2Vec...
Combined feature shape: (45930, 400)


### Combine TF-IDF + embeddings

In [11]:
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=50000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf   = tfidf.transform(X_val)

In [12]:
# Convert dense embeddings to sparse and concatenate
X_train_both = hstack([X_train_tfidf, csr_matrix(X_train_combined)])
X_val_both   = hstack([X_val_tfidf,   csr_matrix(X_val_combined)])

In [13]:
y_val_int = [1 if y == "M" else 0 for y in y_val]
y_train_int = [1 if y == "M" else 0 for y in y_train]

### Logistic Regression

In [19]:
print("\n--- Logistic Regression ---")
lr = LogisticRegression(
    class_weight={0: 1, 1: 6},
    max_iter=1000,
    C=1.0,
    solver='lbfgs'
)
lr.fit(X_train_both, y_train)

lr_preds = lr.predict(X_val_both)
lr_probs = lr.predict_proba(X_val_both)[:, 1]   # P(Mitterrand)

print(classification_report(y_val, lr_preds, target_names=["Chirac", "Mitterrand"]))
print(f"Accuracy: {accuracy_score(y_val, lr_preds):.4f}")



--- Logistic Regression ---
              precision    recall  f1-score   support

      Chirac       0.95      0.90      0.92      9978
  Mitterrand       0.51      0.70      0.59      1505

    accuracy                           0.87     11483
   macro avg       0.73      0.80      0.76     11483
weighted avg       0.89      0.87      0.88     11483

Accuracy: 0.8709


In [20]:
evaluate(y_val, lr_probs,  "FastText + LR")


────────────────────────────────────────
  FastText + LR
────────────────────────────────────────
  F1  (Mitterrand): 0.5873
  AUC (ROC):        0.8866
  AP  (PR curve):   0.6340
────────────────────────────────────────
              precision    recall  f1-score   support

      Chirac       0.95      0.90      0.92      9978
  Mitterrand       0.51      0.70      0.59      1505

    accuracy                           0.87     11483
   macro avg       0.73      0.80      0.76     11483
weighted avg       0.89      0.87      0.88     11483



{'f1': 0.5872529919287504, 'auc': 0.8865861706385276, 'ap': 0.6340079760087034}

### Train SVM on combined vectors

In [22]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

svm_both = CalibratedClassifierCV(
    LinearSVC(class_weight={0: 1, 1: 6}, max_iter=2000),
    cv=3, method='sigmoid'
)
svm_both.fit(X_train_both, y_train)

preds = svm_both.predict(X_val_both)
probs = svm_both.predict_proba(X_val_both)[:, 1]

print(classification_report(y_val, preds, target_names=["Chirac", "Mitterrand"]))
print(f"Accuracy: {accuracy_score(y_val, preds):.4f}")


              precision    recall  f1-score   support

      Chirac       0.92      0.98      0.95      9978
  Mitterrand       0.77      0.41      0.53      1505

    accuracy                           0.91     11483
   macro avg       0.84      0.69      0.74     11483
weighted avg       0.90      0.91      0.89     11483

Accuracy: 0.9067


In [23]:
evaluate(y_val, probs, "FastText + SVM")


────────────────────────────────────────
  FastText + SVM
────────────────────────────────────────
  F1  (Mitterrand): 0.5341
  AUC (ROC):        0.8867
  AP  (PR curve):   0.6594
────────────────────────────────────────
              precision    recall  f1-score   support

      Chirac       0.92      0.98      0.95      9978
  Mitterrand       0.77      0.41      0.53      1505

    accuracy                           0.91     11483
   macro avg       0.84      0.69      0.74     11483
weighted avg       0.90      0.91      0.89     11483



{'f1': 0.5341452805567638, 'auc': 0.8867324725692204, 'ap': 0.65941124073069}

In [24]:
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import matplotlib.pyplot as plt

# After fitting any model, always calibrate:
calibrated = CalibratedClassifierCV(svm_both, cv=3, method='isotonic')  
# use 'isotonic' (more flexible) instead of 'sigmoid' when you have enough data
calibrated.fit(X_train_both, y_train)


,"estimator estimator: estimator instance, default=NoneThe classifier whose output need to be calibrated to provide moreaccurate `predict_proba` outputs. The default classifier isa :class:`~sklearn.svm.LinearSVC`... versionadded:: 1.2",CalibratedCla...ax_iter=2000))
,"method method: {'sigmoid', 'isotonic', 'temperature'}, default='sigmoid'The method to use for calibration. Can be:- 'sigmoid', which corresponds to Platt's method (i.e. a binary logistic regression model).- 'isotonic', which is a non-parametric approach.- 'temperature', temperature scaling.Sigmoid and isotonic calibration methods natively support only binaryclassifiers and extend to multi-class classification using a One-vs-Rest (OvR)strategy with post-hoc renormalization, i.e., adjusting the probabilities aftercalibration to ensure they sum up to 1.In contrast, temperature scaling naturally supports multi-class calibration byapplying `softmax(classifier_logits/T)` with a value of `T` (temperature)that optimizes the log loss.For very uncalibrated classifiers on very imbalanced datasets, sigmoidcalibration might be preferred because it fits an additional interceptparameter. This helps shift decision boundaries appropriately when theclassifier being calibrated is biased towards the majority class.Isotonic calibration is not recommended when the number of calibration samplesis too low ``(≪1000)`` since it then tends to overfit... versionchanged:: 1.8 Added option 'temperature'.",'isotonic'
,"cv cv: int, cross-validation generator, or iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross-validation,- integer, to specify the number of folds.- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if ``y`` is binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used. If ``y`` isneither binary nor multiclass, :class:`~sklearn.model_selection.KFold`is used.Refer to the :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors.Base estimator clones are fitted in parallel across cross-validationiterations.See :term:`Glossary ` for more details... versionadded:: 0.24",None
,"ensemble ensemble: bool, or ""auto"", default=""auto""Determines how the calibrator is fitted.""auto"" will use `False` if the `estimator` is a:class:`~sklearn.frozen.FrozenEstimator`, and `True` otherwise.If `True`, the `estimator` is fitted using training data, andcalibrated using testing data, for each `cv` fold. The final estimatoris an ensemble of `n_cv` fitted classifier and calibrator pairs, where`n_cv` is the number of cross-validation folds. The output is theaverage predicted probabilities of all pairs.If `False`, `cv` is used to compute unbiased predictions, via:func:`~sklearn.model_selection.cross_val_predict`, which are thenused for calibration. At prediction time, the classifier used is the`estimator` trained on all the data.Note that this method is also internally implemented in:mod:`sklearn.svm` estimators with the `probabilities=True` parameter... versionadded:: 0.24.. versionchanged:: 1.6 `""auto""` option is added and is the default.",'auto'
,"penalty penalty: {'l1', 'l2'}, default='l2'Specifies the norm used in the penalization. The 'l2'penalty is the standard used in SVC. The 'l1' leads to ``coef_``vectors that are sparse.",'l2'
,"loss loss: {'hinge', 'squared_hinge'}, default='squared_hinge'Specifies the loss function. 'hinge' is the standard SVM loss(used e.g. by the SVC class) while 'squared_hinge' is thesquare of the hinge loss. The combination of ``penalty='l1'``and ``loss='hinge'`` is not supported.",'squared_hinge'
,"dual dual: ""auto"" or

In [25]:
evaluate(y_val, probs, "FastText + SVM")


────────────────────────────────────────
  FastText + SVM
────────────────────────────────────────
  F1  (Mitterrand): 0.5341
  AUC (ROC):        0.8867
  AP  (PR curve):   0.6594
────────────────────────────────────────
              precision    recall  f1-score   support

      Chirac       0.92      0.98      0.95      9978
  Mitterrand       0.77      0.41      0.53      1505

    accuracy                           0.91     11483
   macro avg       0.84      0.69      0.74     11483
weighted avg       0.90      0.91      0.89     11483



{'f1': 0.5341452805567638, 'auc': 0.8867324725692204, 'ap': 0.65941124073069}